# Fed-ISIC2019 6-Client — Kaggle 확장 실험 A (학습량 증가 18라운드 + FedBN)
`NOWEAZ/fedisic-federated-framework` 브랜치 `feature/v2-extended-experiments`(`fix/notebook-source-resume-state`에서 갈라진 확장 코드) 최신 커밋을 사용하고, 기존 3시드 결과는 브랜치 `results/fed-isic-6client-1101`에서 읽습니다.

**이번 커밋에서 돌리는 것 (대략 9시간 예상, 12시간 한도 안). 학습량 증가(E4)와 FedBN(E3)은 서로 다른 설정을 섞지 않고 각각 따로 돌립니다.**
- E4 학습량 증가: FedAvg와 FedProx(μ=0.1)를 **18라운드**(공식 9라운드의 2배)로, 시드 42/100/2026, test는 `--final_test`로 1회. 학습률·옵티마이저 등 다른 설정은 기준선과 동일.
- E3 FedBN: 시드 42, 100, 2026 (공식 9라운드, 센터별 BN, `--final_test`)
- 분석: 기존 9라운드 결과와 같은 시드끼리 짝지어 `analysis/analyze_extended.py` 실행

**실행 전 설정(우측 Settings):** Accelerator = GPU (T4 x2 또는 P100), Internet = On. 처음에는 위에서부터 한 셀씩 실행해 오류가 없는지 확인한 뒤 `Save Version → Save & Run All (Commit)`.
결과: `/kaggle/working/results_v2.tar.gz` (Output 탭).

In [ ]:
import os, subprocess, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())
!nvidia-smi
assert torch.cuda.is_available(), "GPU가 꺼져 있습니다. Settings > Accelerator에서 GPU를 선택하세요."

In [ ]:
# 1. 프로젝트 코드(확장 브랜치)와 FLamby(고정 커밋) 준비
REPO_URL = "https://github.com/NOWEAZ/fedisic-federated-framework.git"
REPO_BRANCH = "feature/v2-extended-experiments"
RESULTS_BRANCH = "results/fed-isic-6client-1101"   # 기존 6센터 3시드 결과(시드 41 탐색 + 42/100/2026 test 예측)
RESULTS_DIR = "/kaggle/working/RESULTS_OLD"
PROJECT_DIR = "/kaggle/working/CDII"
FLAMBY_DIR = "/kaggle/working/FLamby"
FLAMBY_COMMIT = "edacf54d5211520583b0133d55ac39b6fda8324b"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {REPO_BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git log --oneline -1
# 확장 기능이 들어 있는 브랜치인지 확인
assert "--keep_optimizer" in open("main.py", encoding="utf-8").read() and os.path.exists("analysis/analyze_extended.py"), "확장 코드가 아닙니다"
if not os.path.exists(RESULTS_DIR):
    !git clone --depth 1 -b {RESULTS_BRANCH} {REPO_URL} {RESULTS_DIR}
REPO_OLD = os.path.join(RESULTS_DIR, "results", "fed_isic_6client_kaggle_1101")
assert os.path.exists(os.path.join(REPO_OLD, "stage1_summary.csv")), "기존 결과 폴더를 찾지 못했습니다"
!pip install -q albumentations==1.3.1
!pip install -q -r requirements.txt
if not os.path.exists(FLAMBY_DIR):
    !git clone https://github.com/owkin/FLamby.git {FLAMBY_DIR}
    !cd {FLAMBY_DIR} && git checkout {FLAMBY_COMMIT}
!cd {FLAMBY_DIR} && pip install -q -e ".[isic2019]"
import sys
if FLAMBY_DIR not in sys.path: sys.path.insert(0, FLAMBY_DIR)
import flamby; print("FLamby:", flamby.__file__)

In [ ]:
# 2. 데이터 준비. 우선순위: (a) /kaggle/input 에 붙인 전처리 데이터셋 -> (b) ISIC2019 다운로드 + FLamby 전처리
#    (ISIC 2019 / HAM10000 데이터는 CC-BY-NC 라이선스입니다.)
import glob, shutil
SAVE_V2 = "/kaggle/working/results_v2"      # E4·E3 결과 (test 1회 포함)
DATA_PATH = "/kaggle/temp/isic_data"   # 임시 디스크(출력에 포함되지 않음, 용량 여유)
for d in (SAVE_V2, DATA_PATH): os.makedirs(d, exist_ok=True)
PRE = "ISIC_2019_Training_Input_preprocessed"
PRE_DIR = os.path.join(DATA_PATH, PRE)

def have_data(): return os.path.isdir(PRE_DIR) and len(os.listdir(PRE_DIR)) > 20000

if not have_data():
    tars = glob.glob("/kaggle/input/**/*isic*preprocessed*.tar", recursive=True)
    dirs = glob.glob(f"/kaggle/input/**/{PRE}", recursive=True)
    if tars:
        print("입력 데이터셋의 tar를 풉니다:", tars[0])
        subprocess.run(["tar", "-xf", tars[0], "-C", DATA_PATH], check=True)
    elif dirs:
        print("입력 데이터셋 폴더를 복사합니다:", dirs[0])
        shutil.copytree(dirs[0], PRE_DIR)
if not have_data():
    print("전처리 데이터가 없어 ISIC2019를 다운로드·전처리합니다 (약 9GB, 수십 분).")
    SCRIPT_DIR = os.path.join(FLAMBY_DIR, "flamby", "datasets", "fed_isic2019", "dataset_creation_scripts")
    %cd {SCRIPT_DIR}
    # ISIC가 2019 파일 주소를 옮겨서 FLamby 스크립트의 옛 주소를 치환하고, 약관 질문(y/n)에는 y를 자동 전달(chang 허락 1건), 압축 해제 로그는 줄임
    !sed -i 's#isic-challenge-data.s3.amazonaws.com/2019/#isic-archive.s3.amazonaws.com/challenges/2019/#g' download_isic.py
    !sed -i 's#"unzip #"unzip -q -o #' download_isic.py
    !yes y | python download_isic.py --output-folder "{DATA_PATH}"
    !python resize_images.py
    %cd {PROJECT_DIR}
print("전처리 이미지 수:", len(os.listdir(PRE_DIR)) if os.path.isdir(PRE_DIR) else 0, "(정상: 약 23,247)")
assert have_data(), "데이터 준비 실패"

In [ ]:
# 3. 스모크 테스트: 6개 센터 로더와 학습/검증/테스트 표본 수
%cd {PROJECT_DIR}
from data_utils import get_dataloaders, BATCH_SIZE, LR, NUM_CLIENTS, NUM_EPOCHS_POOLED, get_nb_max_rounds
cl, cen, val, test = get_dataloaders(batch_size=BATCH_SIZE, num_workers=2, data_path=DATA_PATH, seed=42, val_fraction=0.1, split_seed=0)
for k in range(NUM_CLIENTS):
    print(k, len(cl[k].dataset), len(val.client_eval_loaders[k].dataset), len(test.client_eval_loaders[k].dataset))
print("BATCH", BATCH_SIZE, "LR", LR, "pooled epochs", NUM_EPOCHS_POOLED, "FL rounds", get_nb_max_rounds(100, batch_size=BATCH_SIZE))
# 기대값(train/val/test): C0 8937/993/2483, C1 2846/317/791, C2 2423/268/672, C3 1625/182/452, C4 589/66/164, C5 313/38/88
del cl, cen, val, test

In [ ]:
# 4. 공통 설정 (학습률·옵티마이저 등은 공식 설정 그대로, 라운드 수만 바꿈)
from data_utils import BATCH_SIZE, LR, get_nb_max_rounds
NUM_UPDATES = 100
FL_ROUNDS = get_nb_max_rounds(NUM_UPDATES, batch_size=BATCH_SIZE)   # 공식 9
EXT_ROUNDS = 2 * FL_ROUNDS                                          # 확장 18
NUM_WORKERS = 4
EXT_SEEDS = [42, 100, 2026]
FEDBN_SEEDS = [42, 100, 2026]
MU = 0.1   # 기존 6센터 3시드 결과에서 선정된 값. 이번에는 바꾸지 않음(μ 재확인은 다음 커밋)
BASE_ARGS = (f'--batch_size {BATCH_SIZE} --lr {LR} --num_workers {NUM_WORKERS} --val_fraction 0.1 --split_seed 0 '
             f'--data_path "{DATA_PATH}" --resume')
print(FL_ROUNDS, EXT_ROUNDS, BASE_ARGS)

In [ ]:
# 5. E4: 학습량 증가 (18라운드). 파일 이름에 _r18이 붙어 9라운드 결과와 구분됨
%cd {PROJECT_DIR}
for s in EXT_SEEDS:
    print(f"\n===== [E4] FedAvg {EXT_ROUNDS} rounds seed={s} =====")
    !python main.py --method fedavg --rounds {EXT_ROUNDS} --tag r18 --num_updates {NUM_UPDATES} --seed {s} {BASE_ARGS} --save_dir "{SAVE_V2}" --final_test
    print(f"\n===== [E4] FedProx mu={MU} {EXT_ROUNDS} rounds seed={s} =====")
    !python main.py --method fedprox --mu {MU} --rounds {EXT_ROUNDS} --tag r18 --num_updates {NUM_UPDATES} --seed {s} {BASE_ARGS} --save_dir "{SAVE_V2}" --final_test

In [ ]:
# 6. E3: FedBN 5시드 (공식 9라운드). BN 층은 집계하지 않고 센터별로 유지. 학습 후 test는 --final_test로 1회만 평가
%cd {PROJECT_DIR}
for s in FEDBN_SEEDS:
    print(f"\n===== [E3] FedBN seed={s} =====")
    !python main.py --method fedbn --rounds {FL_ROUNDS} --num_updates {NUM_UPDATES} --seed {s} {BASE_ARGS} --save_dir "{SAVE_V2}" --final_test

In [ ]:
# 7. 분석: 기존 9라운드 3시드(Centralized, FedAvg, FedProx μ=0.1)와 같은 시드끼리 짝지어 확장 분석
#    FedBN은 센터별 BN 모델로 평가한 것이라 FedAvg/FedProx(서버 모델 하나)와 평가 대상이 다르다는 점에 유의.
import shutil, glob, json
import pandas as pd
AN_IN = "/kaggle/working/analysis_inputs"; AN_OUT = "/kaggle/working/analysis_extended"
os.makedirs(AN_IN, exist_ok=True)
for pat in ("centralized", "fedavg", "fedprox_mu0.1"):
    for s in (42, 100, 2026):
        shutil.copy(os.path.join(REPO_OLD, f"{pat}_seed{s}_final_test_preds.npz"), AN_IN)
for pat in ("fedavg_r18", "fedprox_mu0.1_r18", "fedbn"):
    for s in (42, 100, 2026):   # 짝지은 비교용: 기존 3시드만 (FedBN의 7, 1234는 results_v2에만 보관)
        shutil.copy(os.path.join(SAVE_V2, f"{pat}_seed{s}_final_test_preds.npz"), AN_IN)
!python analysis/analyze_extended.py --results_dir "{AN_IN}" --out_dir "{AN_OUT}" --pairs fedavg_r18:fedavg fedprox_mu0.1_r18:fedprox_mu0.1 fedprox_mu0.1_r18:fedavg_r18 fedbn:fedavg fedbn:fedprox_mu0.1
display(pd.read_csv(os.path.join(AN_OUT, "paired_summary.csv")))
display(pd.read_csv(os.path.join(AN_OUT, "bootstrap_ci_overall.csv")))

In [ ]:
# 8. 요약과 묶기
rows = []
for p in sorted(glob.glob(os.path.join(SAVE_V2, "*_final_test.json"))):
    d = json.load(open(p))["final"]
    rows.append({"file": os.path.basename(p), **{k: d[k] for k in ("balanced_acc", "macro_f1", "ece", "bal_acc_gap", "bal_acc_std")}})
fb = pd.DataFrame(rows); display(fb)
fb["method"] = fb["file"].str.replace(r"_seed\d+_final_test\.json", "", regex=True)
display(fb.drop(columns="file").groupby("method").agg(["mean", "std"]))
!cd /kaggle/working && tar -czf results_v2.tar.gz results_v2 analysis_extended && ls -la results_v2.tar.gz